# Jeveration — single-pass 512×512 image generation on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Jeveration-single-pass-image-generation.ipynb)

[`linlinlinlinsdada/Jeveration`](https://huggingface.co/linlinlinlinsdada/Jeveration) is LoRA adapters (rank 256) on a **frozen** [`Qwen/Qwen3.5-4B`](https://huggingface.co/Qwen/Qwen3.5-4B) backbone that returns a whole **512×512** image as one joint decision in a **single forward pass**. [Apache-2.0](https://huggingface.co/linlinlinlinsdada/Jeveration) on these weights and the [Carol-lyh/Jeveration](https://github.com/Carol-lyh/Jeveration) code; public (not gated). The files are PyTorch state dicts (`jeveration.pt` **4.0 GB**, `decoder512.pt` **1.2 GB**, `source_decoder512.pt` **1.3 GB`), not a `transformers` / `diffusers` checkpoint — load them with `JeverationPipeline`.

**Why this model.** Autoregressive and diffusion generators take many sequential steps. Jeveration samples all **128** image codes in parallel, conditioned on one shared random state. One checkpoint does text-to-image, image-to-image (HED edges or depth → image, colorize, anime / ghibli / watercolor), and dense prediction (13-class NYUv2 segmentation, depth, surface normals). The paper and project page time **78 ms** per 512 px image on one **H200** (0.74 GenEval; 122× vs Janus-Pro-7B on that table). Those are H200 numbers, not this T4.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/Jeveration-single-pass-image-generation.md) for when this notebook is a good fit and when it is not.

**Licence:** Apache-2.0 on the Jeveration weights and code. Runtime downloads keep their own licences: Qwen3.5-4B, TA-TiTok (`turkeyju/tokenizer_tatitok_bl128_vq`), CLIP ViT-L/14-336 (`timm/vit_large_patch14_clip_336.openai`).

**Runtime:** *Runtime → Change runtime type → T4 GPU*. A free T4 has ~15 GB and **no native bfloat16**. The repo loads the backbone in bfloat16 and keeps extra heads in float32; that does not fit here. This notebook patches the clone to **float16**, loads **only the decoder each demo needs**, and frees the unused decoder between demos. First run also downloads the backbone (~9.3 GB shards), TA-TiTok (`model.safetensors` 1.6 GB; only the codebook is kept), and CLIP (`open_clip_model.safetensors` 1.7 GB; the vision tower is dropped). **Do not load both 512-px decoders at once.**

**Not in this notebook:** the Gradio demo (`app.py`), HED/depth-from-photo (`--from-photo`), or a rerun of the H200 GenEval / NYUv2 tables.

**Links:** [model card](https://huggingface.co/linlinlinlinsdada/Jeveration) · [GitHub](https://github.com/Carol-lyh/Jeveration) · [paper](https://arxiv.org/abs/2611.24738) · [project page](https://carol-lyh.github.io/Jeveration/) · [demo Space](https://huggingface.co/spaces/linlinlinlinsdada/Jeveration) · [announcement](https://x.com/carol00892/status/2123808797009543293) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/Jeveration-single-pass-image-generation.md)


## 0. Early GPU check

Print `torch.cuda.is_available()`, the GPU name, and free VRAM before installing anything. Need a CUDA GPU (T4 is the free-Colab target). A restart after the install cell drops this state; section 2 prints it again.


In [ ]:
import shutil

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 on this GPU; this notebook will load float16.")
else:
    print("GPU name: none")
    print("VRAM free: n/a")
    print("Select Runtime → Change runtime type → T4 GPU, then rerun from the top.")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 1. Install Jeveration (keep Colab's torch)

The [model card](https://huggingface.co/linlinlinlinsdada/Jeveration) / [README](https://github.com/Carol-lyh/Jeveration) install is:

```bash
git clone https://github.com/Carol-lyh/Jeveration && cd Jeveration
pip install -r requirements.txt
hf download linlinlinlinsdada/Jeveration --local-dir weights
```

`requirements.txt` pins `torch==2.7.1` and `transformers==5.12.1`. **Do not reinstall torch** — Colab already ships a CUDA wheel; forcing 2.7.1 often drops GPU support. Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled torch.

This cell clones commit `e0cc08bb8fa4a0c0c060cd189e55901010c2c92c`, installs the remaining pins (`transformers==5.12.1`, `open_clip_torch==3.3.0`, `omegaconf==2.3.0`, `einops==0.8.1`, `flash-linear-attention==0.5.2`), and writes a small T4 patch: backbone **float16** on CUDA, LoRA tensors created in the backbone dtype, and `torch.autocast` **float16** instead of bfloat16.

Weights are downloaded later, per demo — not the full 6.5 GB snapshot up front.

**Do not import `jeveration` in this cell.** Colab has already imported its preinstalled transformers. After reconnect, skip this cell and continue from section 2.


In [ ]:
import importlib.metadata as md
import os
import signal
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

JEV_COMMIT = "e0cc08bb8fa4a0c0c060cd189e55901010c2c92c"
JEV_DIR = Path("/content/Jeveration") if Path("/content").is_dir() else Path("Jeveration").resolve()
INSTALL_TAG = f"jeveration-{JEV_COMMIT[:12]}-tf5.12.1"
MARKER = (
    Path("/content/.jeveration_nb_install_ok")
    if Path("/content").is_dir()
    else Path(".jeveration_nb_install_ok")
)
already_restarted = MARKER.exists() and MARKER.read_text().strip() == INSTALL_TAG

print("python", sys.version.split()[0])
try:
    import torch

    print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
except ImportError:
    print("torch not imported yet")

# Public weights: skip Colab userdata.get("HF_TOKEN") popups.
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

if not (JEV_DIR / ".git").is_dir():
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "https://github.com/Carol-lyh/Jeveration.git", str(JEV_DIR)]
    )
head = subprocess.check_output(["git", "-C", str(JEV_DIR), "rev-parse", "HEAD"], text=True).strip()
if head != JEV_COMMIT:
    subprocess.check_call(["git", "-C", str(JEV_DIR), "fetch", "--depth", "1", "origin", JEV_COMMIT])
    subprocess.check_call(["git", "-C", str(JEV_DIR), "checkout", "--detach", JEV_COMMIT])
print("Jeveration commit", subprocess.check_output(["git", "-C", str(JEV_DIR), "rev-parse", "HEAD"], text=True).strip())


def patch_jeveration_for_t4(root: Path) -> None:
    """Repo defaults are bf16 + CPU-then-.to(cuda). A T4 has no native bf16 and ~15 GB VRAM."""
    backbone = root / "jeveration" / "backbone.py"
    text = backbone.read_text()
    text = text.replace(
        "def load_backbone(source=None, dtype=torch.bfloat16):",
        "def load_backbone(source=None, dtype=torch.float16):",
    )
    text = text.replace(
        "full = AutoModelForImageTextToText.from_pretrained(repo, revision=revision, dtype=dtype)",
        "full = AutoModelForImageTextToText.from_pretrained(\n"
        "        repo, revision=revision, dtype=dtype, device_map=\"cuda\", low_cpu_mem_usage=True)",
    )
    text = text.replace(
        "self.lora_A = nn.Parameter(torch.zeros(rank, base.in_features))\n"
        "        self.lora_B = nn.Parameter(torch.zeros(base.out_features, rank))",
        "kw = dict(dtype=base.weight.dtype, device=base.weight.device)\n"
        "        self.lora_A = nn.Parameter(torch.zeros(rank, base.in_features, **kw))\n"
        "        self.lora_B = nn.Parameter(torch.zeros(base.out_features, rank, **kw))",
    )
    backbone.write_text(text)

    for rel in (
        "jeveration/pipeline.py",
        "jeveration/model.py",
        "jeveration/tokenizer.py",
    ):
        path = root / rel
        path.write_text(path.read_text().replace("torch.bfloat16", "torch.float16"))

    pipe = root / "jeveration" / "pipeline.py"
    ptxt = pipe.read_text()
    ptxt = ptxt.replace(
        "lat_mean, lat_std = self.model.load_weights(weights)\n        tok = load_text_tokenizer(backbone)",
        "lat_mean, lat_std = self.model.load_weights(weights)\n"
        "        self.model.to(dtype=torch.float16)\n"
        "        tok = load_text_tokenizer(backbone)",
    )
    needle = (
        "            if source_decoder:\n"
        "                self.codec.load_source_decoder(source_decoder)"
    )
    if needle in ptxt:
        ptxt = ptxt.replace(
            needle,
            needle + "\n            self.codec.to(dtype=torch.float16)",
        )
    pipe.write_text(ptxt)
    print("T4 patch: float16 backbone, LoRA, autocast, and extra heads.")


patch_jeveration_for_t4(JEV_DIR)

# Card: pip install -r requirements.txt. Keep Colab's CUDA torch. Use %pip, not python -I.
get_ipython().run_line_magic(
    "pip",
    'install -q "transformers==5.12.1" accelerate "open_clip_torch==3.3.0" "omegaconf==2.3.0" '
    '"einops==0.8.1" "flash-linear-attention==0.5.2" "safetensors>=0.5" '
    '"huggingface_hub>=0.34" matplotlib pillow',
)

print()
print("JEV_DIR", JEV_DIR)
for pkg in (
    "transformers",
    "open_clip_torch",
    "omegaconf",
    "einops",
    "flash_linear_attention",
    "safetensors",
    "huggingface_hub",
):
    try:
        print(pkg, md.version(pkg))
    except md.PackageNotFoundError:
        print(pkg, "not installed")

MARKER.write_text(INSTALL_TAG)
if already_restarted:
    print("Install marker already set; not restarting again. Continue from section 2.")
else:
    print()
    print("=" * 72)
    print("Restarting the session so transformers==5.12.1 is imported.")
    print("After reconnect: skip this cell and run from section 2 (runtime check).")
    print("A 'session crashed' or 'Operation cancelled' message here is the restart,")
    print("not a failed install — pip already finished (versions printed above).")
    print("=" * 72)
    os.kill(os.getpid(), signal.SIGKILL)


> The install cell **restarts the session for you**. After reconnect, skip it and continue from section 2.
> A "session crashed" message after install is the restart, not a failed pip. If you restarted by hand instead: *Runtime → Restart session*, then this next cell.


## 2. Runtime check after restart

Confirm `transformers==5.12.1`, CUDA, device name, and free VRAM. T4 (`sm_75`) has no native bfloat16 — **float16** is what this notebook loads.


In [ ]:
import gc
import os
import shutil
import sys
import time
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

import torch
import transformers
from packaging.version import Version

JEV_COMMIT = "e0cc08bb8fa4a0c0c060cd189e55901010c2c92c"
JEV_DIR = Path("/content/Jeveration") if Path("/content").is_dir() else Path("Jeveration").resolve()
WEIGHTS_DIR = Path("/content/jeveration_weights") if Path("/content").is_dir() else Path("jeveration_weights")
SAMPLE_DIR = Path("/content/jeveration_samples") if Path("/content").is_dir() else Path("jeveration_samples")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

if str(JEV_DIR) not in sys.path:
    sys.path.insert(0, str(JEV_DIR))

tf_ver = Version(transformers.__version__)
if tf_ver < Version("5.12.1"):
    raise SystemExit(
        f"transformers {transformers.__version__} is loaded; Jeveration's repo pins 5.12.1. "
        "Run the install cell, then Runtime → Restart session."
    )

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime → Change runtime type → T4 GPU, then rerun from the GPU-check cell.")

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

free_b, total_b = torch.cuda.mem_get_info()
major, minor = torch.cuda.get_device_capability()
print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")
print("JEV_DIR", JEV_DIR)
assert (JEV_DIR / "jeveration" / "pipeline.py").is_file(), (
    f"{JEV_DIR} has no jeveration package; rerun the install cell."
)
assert "dtype=torch.float16" in (JEV_DIR / "jeveration" / "backbone.py").read_text(), (
    "T4 float16 patch missing from backbone.py; rerun the install cell."
)

NATIVE_BF16 = major >= 8
DEVICE = "cuda"
DTYPE = torch.float16
print("DEVICE", DEVICE, "| DTYPE", DTYPE)
if not NATIVE_BF16:
    print("No native bfloat16; loading float16. Repo code was patched in the install cell.")
if NATIVE_BF16:
    print("This GPU can run bfloat16; this notebook still uses float16 so a T4 recipe stays consistent.")


def print_vram(label=""):
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_b, total_b = torch.cuda.mem_get_info()
    tag = f"{label} | " if label else ""
    print(
        f"{tag}allocated {allocated:.2f} GB | peak {peak:.2f} GB | "
        f"free {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB"
    )


print_vram("after runtime check")


## 3. Helpers and sample media

`JeverationPipeline` wants uint8 RGB arrays. Official examples come from the [GitHub repo](https://github.com/Carol-lyh/Jeveration/tree/main/examples) (`gray_pancakes.jpg`, `room_1.png`). Each download has a COCO fallback. No secrets.


In [ ]:
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import numpy as np
import requests
from IPython.display import display
from PIL import Image

from huggingface_hub import hf_hub_download

HF_REPO = "linlinlinlinsdada/Jeveration"
EX_COMMIT = JEV_COMMIT
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}
GRAY_URLS = [
    f"https://raw.githubusercontent.com/Carol-lyh/Jeveration/{EX_COMMIT}/examples/gray_pancakes.jpg",
    "https://github.com/Carol-lyh/Jeveration/raw/main/examples/gray_pancakes.jpg",
    f"https://cdn.jsdelivr.net/gh/Carol-lyh/Jeveration@{EX_COMMIT}/examples/gray_pancakes.jpg",
]
ROOM_URLS = [
    f"https://raw.githubusercontent.com/Carol-lyh/Jeveration/{EX_COMMIT}/examples/room_1.png",
    "https://github.com/Carol-lyh/Jeveration/raw/main/examples/room_1.png",
    f"https://cdn.jsdelivr.net/gh/Carol-lyh/Jeveration@{EX_COMMIT}/examples/room_1.png",
]
COCO_INDOOR = "http://images.cocodataset.org/val2017/000000000139.jpg"
COCO_CATS = "http://images.cocodataset.org/val2017/000000039769.jpg"
T2I_PROMPT = "A lone pine tree on a small island in a calm lake at dawn."
COLORIZE_CAPTION = None  # colorize uses a fixed instruction; no caption


def fetch_bytes(url, attempts=4, timeout=120):
    last = None
    for attempt in range(attempts):
        try:
            response = requests.get(url, headers=UA, timeout=timeout)
            if response.status_code in (429, 500, 502, 503, 504) and attempt < attempts - 1:
                time.sleep(2 ** (attempt + 1))
                continue
            response.raise_for_status()
            return response.content
        except Exception as exc:
            last = exc
            try:
                req = Request(url, headers=UA)
                with urlopen(req, timeout=timeout) as fh:
                    return fh.read()
            except Exception as exc2:
                last = exc2
                time.sleep(1)
    raise RuntimeError(f"download failed: {url}") from last


def fetch_to(path, urls, fallback_urls=()):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        print("have", path.name, f"({path.stat().st_size} bytes)")
        return path
    last_error = None
    for url in list(urls) + list(fallback_urls):
        try:
            data = fetch_bytes(url)
            path.write_bytes(data)
            print("saved", path.name, f"({path.stat().st_size} bytes) from {url.split('/')[-1]}")
            return path
        except Exception as exc:
            last_error = exc
            print(f"failed {url}: {type(exc).__name__}: {exc}")
    raise RuntimeError(f"could not download {path.name}") from last_error


def show_row(images, titles=None, max_side=360):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4.2))
    if n == 1:
        axes = [axes]
    for i, img in enumerate(images):
        arr = np.asarray(img)
        axes[i].imshow(arr)
        axes[i].axis("off")
        if titles:
            axes[i].set_title(titles[i])
    plt.tight_layout()
    display(fig)
    plt.close(fig)


def download_weight(name):
    dest = WEIGHTS_DIR / name
    if dest.exists() and dest.stat().st_size > 0:
        print("have", dest.name, f"({dest.stat().st_size / 1024**3:.2f} GB)")
        return str(dest)
    path = hf_hub_download(HF_REPO, name, local_dir=str(WEIGHTS_DIR))
    print("downloaded", Path(path).name, f"({Path(path).stat().st_size / 1024**3:.2f} GB)")
    return path


gray_path = fetch_to(SAMPLE_DIR / "gray_pancakes.jpg", GRAY_URLS, fallback_urls=[COCO_CATS])
room_path = fetch_to(SAMPLE_DIR / "room_1.png", ROOM_URLS, fallback_urls=[COCO_INDOOR])
print("samples:", gray_path.name, room_path.name)
show_row(
    [Image.open(gray_path).convert("RGB"), Image.open(room_path).convert("RGB")],
    titles=["colorize input", "dense input (indoor)"],
)


## 4. Load Jeveration for text-to-image

Card API: `JeverationPipeline(weights, decoder=..., source_decoder=...)`. On a T4 we pass **only** `jeveration.pt` + `decoder512.pt` (no source decoder). Backbone `Qwen/Qwen3.5-4B` at the card's pinned revision is fetched on first use. Extra heads are cast to float16 after `load_weights` so LoRA does not sit in VRAM as float32.


In [ ]:
from jeveration import JeverationPipeline
from jeveration.images import load_square, to_gray

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print_vram("before load")

weights_path = download_weight("jeveration.pt")
decoder_path = download_weight("decoder512.pt")

pipe = JeverationPipeline(
    weights_path,
    decoder=decoder_path,
    source_decoder=None,
    device=DEVICE,
)
pipe.model.to(dtype=DTYPE)
if pipe.codec is not None:
    pipe.codec.to(dtype=DTYPE)

print("pipeline ready | decoder:", type(pipe.codec.decoder).__name__ if pipe.codec else None)
print("source decoder:", None if pipe.codec is None else type(pipe.codec.source_decoder).__name__ if pipe.codec.source_decoder else None)
print_vram("after T2I load")


## 5. Demo 1 — text-to-image with timing

README prompt: *A lone pine tree on a small island in a calm lake at dawn.* Seed `0`. One forward pass; wall time is this T4, not the paper's **78 ms on one H200**.


In [ ]:
from IPython.display import display

print("prompt:", T2I_PROMPT)
torch.cuda.synchronize()
t0 = time.perf_counter()
images = pipe.text_to_image([T2I_PROMPT], pipe.generator(0))
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

out = images[0]
print(f"output: {out.shape} {out.dtype} | wall time {elapsed:.2f}s")
print(f"finite pixels: {np.isfinite(out.astype(np.float32)).all()}")
show_row([out], titles=[f"text-to-image  seed=0  {elapsed:.2f}s"])
Image.fromarray(out).save(SAMPLE_DIR / "t2i_pine.png")
print_vram("demo 1 text-to-image")


## 6. Demo 2 — colorize (image-to-image)

Free the **plain** 512-px decoder (~1.2 GB) and load **only** `source_decoder512.pt` (~1.3 GB). Task `colorize` reads a grey picture; the README example is `examples/gray_pancakes.jpg`. If that download was the COCO fallback, it is converted to grey with `to_gray`.

Not loaded: HED / depth structure-to-image (needs a caption plus an edge or depth map) or anime / ghibli / watercolor. Those share this source decoder — swap `task=` if you want them after this cell.


In [ ]:
# Drop the text-to-image decoder before the source decoder lands.
if pipe.codec is not None and pipe.codec.decoder is not None:
    del pipe.codec.decoder
    pipe.codec.decoder = None
gc.collect()
torch.cuda.empty_cache()
print_vram("after dropping decoder512")

source_decoder_path = download_weight("source_decoder512.pt")
pipe.codec.load_source_decoder(source_decoder_path)
pipe.codec.to(dtype=DTYPE)
print("source decoder:", type(pipe.codec.source_decoder).__name__)
print_vram("after source decoder load")

gray = to_gray(load_square(str(gray_path)))
print("colorize input:", gray.shape, gray_path.name)

torch.cuda.synchronize()
t0 = time.perf_counter()
colorized = pipe.image_to_image(gray[None], "colorize", generator=pipe.generator(0))
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

print(f"output: {colorized[0].shape} | wall time {elapsed:.2f}s")
show_row([gray, colorized[0]], titles=["greyscale in", f"colorize  {elapsed:.2f}s"])
Image.fromarray(colorized[0]).save(SAMPLE_DIR / "colorize_out.png")
print_vram("demo 2 colorize")


## 7. Demo 3 — dense prediction (seg + depth)

Drop the image codec (CLIP text tower + source decoder). Dense heads live in `jeveration.pt` and do **not** need a 512-px decoder. The card: dense heads were trained on **NYUv2 indoor 4:3** scenes; other content is outside that training. 13 classes: bed, books, ceiling, chair, floor, furniture, objects, picture, sofa, table, tv, wall, window.

The pipeline's default `dense_predict` runs seg, depth, and normals as a batch of 3. This cell runs **seg** then **depth** one task at a time so activations stay smaller on a T4. Normals are skipped here.


In [ ]:
from jeveration import config
from jeveration.dense import depth_view, seg_view

if getattr(pipe, "codec", None) is not None:
    del pipe.codec
    pipe.codec = None
gc.collect()
torch.cuda.empty_cache()
print_vram("after dropping codec")

room = np.asarray(Image.open(room_path).convert("RGB"), dtype=np.uint8)
print("dense input:", room.shape, room_path.name, "| classes:", ", ".join(config.NYU13))

pred = {}
for task in ("seg", "depth"):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = pipe.dense_predict(room[None], tasks=(task,))
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    pred[task] = out[task][0].detach().cpu().numpy()
    print(f"{task}: wall time {elapsed:.2f}s | shape {pred[task].shape}")
    print_vram(f"demo 3 {task}")

seg = pred["seg"].astype(np.int64)
depth = pred["depth"].astype(np.float32)
print("classes present:", ", ".join(config.NYU13[i] for i in np.unique(seg)))
print(f"depth: {depth.min():.2f} to {depth.max():.2f} m (NYUv2 indoor; not a card metric from this T4)")

seg_img = seg_view(seg, room)
depth_img = depth_view(depth)
show_row(
    [room, seg_img, depth_img],
    titles=["photo", "13-class seg", "depth (near=bright)"],
)
Image.fromarray(seg_img).save(SAMPLE_DIR / "dense_seg.png")
Image.fromarray(depth_img).save(SAMPLE_DIR / "dense_depth.png")


## 8. Caveats and cleanup

**What this notebook is for.** The released `JeverationPipeline` on a free T4: one 512×512 text-to-image (timed), colorization via the source decoder, and indoor seg + depth. Peak VRAM is `torch.cuda.max_memory_allocated`.

**What it is not.** The paper's 78 ms / 0.74 GenEval / 67.7 mIoU figures are from the card and project page (H200 / NYUv2), not measurements from this runtime. Structure-to-image (HED, depth) and style transfer share the source decoder loaded in demo 2. The Gradio app is not started.

**Related notebooks:** [WeVisOne-VL 2B](WeVisOne-VL-2B-detect-segment.ipynb) (boxes and cutouts), [MiniCPM-V 4.7 1B](MiniCPM-V-4.7-1B-vision.ipynb) (image Q&A), [LFM3-VL-2B](LFM3-VL-2B-vision.ipynb), [Mirage](Mirage-tiny-vision-structured-extraction.ipynb) (tiny on-device scene JSON).

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
del pipe
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
